# Dato Tabular Temporal y Embbedings de TabPFN — Dataset FDIC RIS

## Ingeniería de características temporales

Ya tenemos los datos tabulares procesados que necesitamos para nuestros modelos de ML. Nuestro panel tabular que servirá como ingestion de datos para el modelo de `TabPFN` (98 variables financieras 
y 206.129 observaciones) no es temporal. Tenemos el panel con 100 variables (`CERT + period + 98 camels features`) y donde cada fila es un banco en un trimestre, i.e., son independientes, luego para que `TabPFN` pueda capturar dinámica temporal, cada fila debe contener información de periodos anteriores comprimida en nuevas columnas. Las transformaciones temporales que aplicamos son:

- __Lags__: (`_lag1`, `_lag2`, `_lag3`): valor de la variable en los 3 trimestres anteriores. Si el ratio de capital cae tres trimestres consecutivos, los lags lo reflejan explícitamente. Se usan 3 lags siguiendo la literatura de _early warning systems_ bancarios, que identifica 3-4 trimestres como horizonte óptimo de anticipación (Cole & Gunther, 1995).

- __Diferencias__: Cambios entre periodos que capturan tendencias inmediatas, más formalmente es un cambio absoluto respecto al trimestre anterior $X_t - X_{t-1}$. Un salto brusco en morosidad es más informativo que su nivel absoluto.

- __Rolling (Media móvil)__: Media de los últimos 3 trimestres que suavizan ruido y capturan tendencia local. `ROA` puede ser volátil; la media móvil de 3 trimestres estabiliza la señal.

- __Tasas de crecimiento__: (`_growth`): cambio relativo $(X_t - X_{t-1}) / |X_{t-1}|$. Normaliza la diferencia por el valor anterior, un cambio de 0.01 en `ROA` tiene distinto significado si `ROA` era 0.1 o 2.0. Útiles pero hay que llevar cuidado ya que podemos tener divisiones por cero cuando $X_{t-1} = 0$.

De esta forma, cada fila contiene una ventana temporal comprimida, lo que convierte el panel tabular (`panel_tabular_camels98.parquet`) en un _panel tabular temporal_. Todas estas nuevas caracteristicas se deben agrupar por `CERT + period`, ya que como cada banco tiene su propia secuencia temporal los `lags` deben calcularse dentro de cada entidad, no globalmente. Esto es  imprescindible para que los `lags` sean correctos. Es decir el resultado final es un panel resultante que no deja de ser tabular, ya que sigue siendo una tabla donde cada fila es un banco en un trimestre. Lo que cambia es que cada fila ahora contiene información implícita de periodos anteriores. El modelo `TabPFN` no sabe que son `lags` o las demas variables temporales que creamos, los trata como _features_ más, pero la información temporal ya está codificada en ellas.

__OBSERVACIÓN:__ Los primeros periodos de cada banco tendrán `NaN` en `lags` y `diferencias`, porque no hay historia disponible para esos trimestres.

Tenemos que crear una funcion en el archivo `scr/data/temporal_features.py` que convierta cualquier panel tabular a un panel con información temporal. Esta función primero ordena el `DataFrame` por `CERT` y luego por `period`, lo cual es fundamental ya que sino, sin este orden los `lags` serían incorrectos, mezclarían periodos de distintos bancos o en orden equivocado. Luego identificamos las columnas sobre las que aplicar las transformaciones excluyendo `CERT` y `period` que son identificadores, no features.

Para cada `variable` y cada `lag`, desplaza la serie temporal `lag` posiciones hacia adelante dentro de cada banco. Por ejemplo, `ROA_lag1` en el trimestre `2017Q2` contiene el valor de `ROA` en `2017Q1`. Los primeros `lag` periodos de cada banco tendrán `NaN` ya que no hay historia disponible. Luego calculamos las diferencias, es decir, el cambio absoluto respecto al trimestre anterior: $X_t - X_{t-1}$. Esto permite capturar tendencias inmediatas, ya que como hemos afirmado, un salto brusco en morosidad es más informativo que su nivel absoluto. El primer periodo de cada banco será `NaN`. Calculamos el cambio relativo (tasa de crecimiento): $(X_t - X_{t-1}) / |X_{t-1}|$. Esto nos permite normalizar la diferencia por el valor anterior, un cambio de 0.01 en `ROA` tiene distinto significado si `ROA` era 0.1 o 2.0. Usamos `replace(0, nan)` evita divisiones por cero. Fianlmente calculamos el _Rolling mean_, que calcula la media móvil de ventana `w` trimestres dentro de cada banco. `min_periods=1` evita `NaN` en los primeros periodos ya que usa los datos disponibles aunque sean menos de `w`, permitiendo suavizar ruido y capturar tendencia local.

Antes de aplicar las transformaciones temporales sobre las 98 variables completas, verificamos empíricamente el impacto en dimensionalidad. `TabPFN v2.5/v2.6` tiene un límite de 500 _features_ en modo estándar, superarlo implica que el modelo no puede procesar el panel completo de forma directa. Trabajamos con un subconjunto piloto de 100 bancos para estimar el resultado sin colapsar la RAM.

In [2]:
import pandas as pd
from pathlib import Path
import sys
from dotenv import load_dotenv
import warnings

# 1. Añadir raíz del proyecto al path (ANTES de importar src)
project_root = Path.cwd().parents[0]
load_dotenv(project_root / ".env")
sys.path.insert(0, str(project_root))

# 2. Ahora sí puedes importar módulos del proyecto
from src.data.temporal_features import build_temporal_features
from src.utils.config import PROCESSED_DIR

# 3. Resto de imports externos

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import importlib

# 4. Prueba con un subconjunto pequeño

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
df = pd.read_parquet(PROCESSED_DIR / "panel_tabular_camels98.parquet")
df_pilot = df[df['CERT'].isin(df['CERT'].unique()[:100])].copy()


panel_piloto = build_temporal_features(
    df_pilot,
    lags=[1, 2, 3],
    windows=[3],
    growth_rate=True
)

print(f"Variables originales: {df_pilot.shape[1]}")
print(f"Variables tras features temporales (todas): {panel_piloto.shape[1]}")
print(f"Nuevas columnas añadidas: {panel_piloto.shape[1] - df_pilot.shape[1]}")
print(f"Memoria estimada panel completo: {panel_piloto.memory_usage(deep=True).sum() / 1e6 * (df['CERT'].nunique() / 100):.0f} MB")
print(f"\nDesglose de columnas nuevas:")
print(f"  Lags (98 vars × 3):    {98*3}")
print(f"  Diferencias (98 vars): {98}")
print(f"  Growth (98 vars):      {98}")
print(f"  Rolling (98 vars):     {98}")
print(f"  Total nuevas:          {98*3 + 98 + 98 + 98}")
print(f"  Total final:           {98 + 98*3 + 98 + 98 + 98}")
print(f"\nSupera el límite de TabPFN (500 features): {98 + 98*3 + 98 + 98 + 98 > 500}")


Variables originales: 100
Variables tras features temporales (todas): 688
Nuevas columnas añadidas: 588
Memoria estimada panel completo: 1255 MB

Desglose de columnas nuevas:
  Lags (98 vars × 3):    294
  Diferencias (98 vars): 98
  Growth (98 vars):      98
  Rolling (98 vars):     98
  Total nuevas:          588
  Total final:           686

Supera el límite de TabPFN (500 features): True


Aplicar las 6 transformaciones sobre las 98 variables genera `588` columnas nuevas, llevando el panel a `688` columnas totales, muy por encima del límite de `TabPFN`. Esto hace necesario acotar las transformaciones temporales a un subconjunto de las variables del panel tabular, ya que no todas las variables `CAMELS` que tenemos necesitan temporalidad.

No todas las variables de nuestro panel tabular requieren `lags`, `diferencias` y `rolling`. Solo aquellas que capturan _dinámica financiera relevante_ se benefician de estas transformaciones:

- Las variables que sí necesitan temporalidad son aquellas relacionadas con ratios de capital, calidad de activos, rentabilidad y liquidez, variables que evolucionan trimestre a trimestre y cuya trayectoria es señal de riesgo.

- No necesitan temporalidad _flags binarios_, proporciones muy estables, variables estructurales y métricas con baja varianza ya que su valor puntual ya es informativo sin historia.

El criterio objetivo para identificar las variables con mayor dinámica relevante es su correlación `punto-biserial` con el evento de quiebra (base de datos de `failure`), calculada en el notebook anterior. Las variables más correlacionadas con failures son precisamente las que exhiben mayor dinámica temporal ratios de capital, calidad de activos y rentabilidad (etiquetas `C`, `A`, `E`, `L`). Se seleccionan las _top 20_ por correlación absoluta.

In [3]:
# Top 20 variables para features temporales
df_corr_final = pd.read_csv(project_root / "docs" / "camels_final_selection.csv")
top20_vars = df_corr_final.head(20)['variable'].tolist()

print(f"\nTop 20 variables seleccionadas para features temporales:")
print(f"{'#':<4} {'Variable':<15} {'Corr':>8} {'CAMELS':>8}  Descripción")
print("-" * 70)
for i, v in enumerate(top20_vars, 1):
    row = df_corr_final[df_corr_final['variable'] == v].iloc[0]
    print(f"  {i:2}. {v:<15} {row['abs_correlation']:>8.4f} {row['camels']:>8}  {row['description'][:35]}")


Top 20 variables seleccionadas para features temporales:
#    Variable            Corr   CAMELS  Descripción
----------------------------------------------------------------------
   1. RBCPCA            0.3020        C  RBC CATEGORY-PCA
   2. AVTANEQ           0.1488        C  AVG TANGIBLE EQUITY-ASSMT
   3. LNAGT1R           0.1380        C  AG LOANS/TIER 1
   4. LNCIT1R           0.1312        C  C&I LOANS/TIER 1
   5. RBCERI            0.1197        E  ELIGIBLE RETAINED INCOME
   6. LNCONT1R          0.1044        C  CONSUMER LOANS/TIER 1
   7. ROEQ              0.0636        C  RETURN ON EQUITY- BANK- QTR
   8. CHBALRCR          0.0508        L  CASH & BAL BS - RC-R COL A
   9. EQ2               0.0505        C  TOTAL BANK EQUITY CAPITAL-CAVG2
  10. NCRELOCR          0.0492        C  N/C HOME EQUITY/HOME EQUITY
  11. EQ5               0.0471        C  TOTAL BANK EQUITY CAPITAL-CAVG5
  12. CHBAL2            0.0399        L  CASH & DUE FROM DEPOS INST-CAVG2
  13. CT1BADJ           

Las 20 variables con mayor correlación `punto-biserial` con `failures` pertenecen predominantemente a los componentes `C` (Capital) y `E` (Earnings) del framework `CAMELS`, consistente con la literatura que identifica la adecuación de capital y la rentabilidad como los predictores más tempranos de quiebra bancaria.

In [ ]:
# Verificación con top 20 — mismo piloto de 100 bancos
panel_top20 = build_temporal_features(
    df_pilot,
    feature_subset=top20_vars,  #
    lags=[1, 2, 3],
    windows=[3],
    growth_rate=True
)

print(f"\nVariables originales:     {df_pilot.shape[1]}")
print(f"Variables con top20:      {panel_top20.shape[1]}")
print(f"Nuevas columnas añadidas: {panel_top20.shape[1] - df_pilot.shape[1]}")
print(f"Dentro del límite TabPFN: {panel_top20.shape[1] <= 500}")


Variables originales:     100
Variables con top20:      220
Nuevas columnas añadidas: 120
Dentro del límite TabPFN: True ✓


Esta decisión tiene además una justificación metodológica adicional: con solo 103 observaciones positivas (quiebras) sobre 206.129, añadir features temporales innecesarias aumenta el riesgo de overfitting sin aportar señal discriminativa.

`ROA` no está en el top 20 por correlación `punto-biserial` con `failures` dado que su correlación es baja en nuestro dataset post-2016 porque los bancos que quiebran en este periodo no siempre muestran ROA negativo antes del cierre (quiebras por liquidez o regulatorias más que por rentabilidad).